In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_groq import ChatGroq
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [14]:
docs = [
    Document(page_content="Artificial intelligence is transforming the software development industry. Recycling plastic bottles helps conserve natural resources. Cloud platforms provide scalable computing resources for businesses."),
    Document(page_content="Regular exercise improves cardiovascular health. Solar panels convert sunlight into electricity. The Internet of Things (IoT) connects everyday devices to the internet."),
    Document(page_content="Adequate sleep is essential for maintaining mental health. Wind turbines generate electricity without producing greenhouse gases. Cybersecurity professionals protect systems from digital threats."),
    Document(page_content="Consuming fruits and vegetables supports a healthy immune system. Forest conservation plays a vital role in preserving biodiversity. Machine learning models can identify patterns in large datasets."),
    Document(page_content="Consuming fruits and vegetables supports a healthy immune system. Forest conservation plays a vital role in preserving biodiversity. Machine learning models can identify patterns in large datasets.")
]

In [15]:
llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

In [16]:
vector_store= FAISS.from_documents(
    documents = docs, 
    embedding =HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

In [17]:
base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [18]:
compressor = LLMChainExtractor.from_llm(llm)

In [19]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [26]:
query = "What are the healthy habits?"

In [27]:
results = compression_retriever.invoke(query)

In [28]:
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}\n")

Result 1: * Regular exercise improves cardiovascular health. 

If no other part of the context seems relevant, and based on the question asked, then only this part appears to relate to "healthy habits".

Result 2: Consuming fruits and vegetables supports a healthy immune system.

Result 3: * Consuming fruits and vegetables supports a healthy immune system.

